# Complaint Dispute Classification - CPU Baseline

This notebook is the CPU reference implementation used in the assignment comparison. It reduces the original project to one reproducible XGBoost workflow so the data preparation, validation results, and benchmark timings can be inspected clearly.

## Scope Of The CPU Notebook

The original project included exploratory analysis, multiple model branches, text-heavy features, and tuning utilities. For this assignment, the notebook is narrowed to the parts needed for a fair CPU-versus-GPU comparison.

### Included In This Version
- Loading the training and inference complaint files.
- Cleaning key fields, standardising missing values, parsing dates, and mapping the target to `0/1`.
- Building a small structured feature set from complaint metadata and narrative availability.
- One-hot encoding the selected categorical inputs.
- Training one XGBoost classifier and evaluating it on a held-out validation split.
- Benchmarking training, validation inference, and external inference across the dataset-size sweep.

### Left Out From The Original Project
- Exploratory-analysis and managerial-insight sections.
- Alternative model families and duplicate experiments.
- TF-IDF text features and larger preprocessing branches.
- Hyperparameter search and threshold-selection logic.
- SHAP interpretation and submission-packaging cells.

This narrower version keeps the prediction task intact while making the benchmark design much easier to follow.

## Setup And Shared Parameters

The imports below define the CPU toolchain used throughout the notebook: pandas and NumPy for tabular preparation, XGBoost for modeling, and a shared set of constants for features, random seeds, and benchmark settings.

In [1]:
from pathlib import Path
from random import Random
from statistics import mean, stdev
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from xgboost import XGBClassifier

RANDOM_STATE = 42
VALIDATION_SIZE = 0.20
SUBSET_FRACTIONS = [0.10, 0.25, 0.50, 0.75, 1.00]
BENCHMARK_REPEATS = 3
PREDICTION_THRESHOLD = 0.50
TARGET_COLUMN = 'Consumer disputed?'
ID_COLUMN = 'Complaint ID'

CATEGORICAL_COLUMNS = [
    'Product',
    'Sub-product',
    'Issue',
    'Sub-issue',
    'Company',
    'State',
    'Tags',
    'Consumer consent provided?',
    'Submitted via',
    'Company response to consumer',
    'Company public response',
    'Timely response?',
]

NUMERIC_COLUMNS = [
    'days_to_company',
    'month_received',
    'weekday_received',
    'narrative_present',
    'narrative_len',
]

MODEL_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS
BENCHMARK_COLUMNS = [
    'fraction',
    'train_rows',
    'validation_rows',
    'inference_rows',
    'benchmark_repeats',
    'train_seconds_mean',
    'train_seconds_std',
    'validation_seconds_mean',
    'validation_seconds_std',
    'inference_seconds_mean',
    'inference_seconds_std',
    'accuracy',
    'precision',
    'recall',
    'f1',
]
MISSING_TOKENS = {
    '', ' ', 'na', 'n/a', 'none', 'null', 'nan',
    'not available', 'not provided', 'unknown'
}
US_STATE_CODES = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','DC','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA','ME',
    'MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI',
    'SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','PR','VI','GU','AS','MP'
}


## Input Files And Run Configuration

Both CSV inputs are read from the notebook directory through relative paths. Keeping the file configuration simple makes the baseline easier to reproduce and ensures that every benchmark run uses the same raw data.

In [2]:
DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'complaints_training.csv'
INFERENCE_PATH = DATA_DIR / 'complaints_inference.csv'

for path in [TRAIN_PATH, INFERENCE_PATH]:
    print(f'{path}: exists={path.exists()}')

complaints_training.csv: exists=True
complaints_inference.csv: exists=True


## Cleaning And Structured Feature Engineering

These helper functions standardise text fields, normalise missing-value tokens, parse the complaint dates, remove duplicate records, and derive a compact set of structured features. The engineered variables focus on operational metadata rather than free-text modeling so the CPU and GPU pipelines remain closely aligned.

In [3]:
def clean_complaints_df(df: pd.DataFrame, is_training: bool) -> pd.DataFrame:
    """Minimal cleaning for the assignment-specific CPU pipeline."""
    out = df.copy()
    out.columns = [col.strip() for col in out.columns]

    object_columns = out.select_dtypes(include=['object']).columns.tolist()
    for col in object_columns:
        cleaned = out[col].apply(lambda value: value.strip() if isinstance(value, str) else value)
        token_mask = cleaned.fillna('').astype(str).str.lower().isin(MISSING_TOKENS)
        out[col] = cleaned.mask(token_mask, np.nan)

    if 'Submitted via' in out.columns:
        submitted_map = {
            'web': 'Web',
            'email': 'Email',
            'fax': 'Fax',
            'phone': 'Phone',
            'referral': 'Referral',
            'postal mail': 'Postal mail',
        }
        submitted = out['Submitted via'].apply(lambda value: value.strip() if isinstance(value, str) else value)
        out['Submitted via'] = submitted.apply(
            lambda value: submitted_map.get(value.lower(), value) if isinstance(value, str) else value
        )

    if 'State' in out.columns:
        state = out['State'].apply(lambda value: value.strip().upper() if isinstance(value, str) else value)
        valid_state = state.isin(US_STATE_CODES)
        out['State'] = state.where(valid_state, np.nan).fillna('Unknown')

    if 'Timely response?' in out.columns:
        timely = out['Timely response?'].apply(lambda value: value.strip().lower() if isinstance(value, str) else value)
        out['Timely response?'] = timely.map({'yes': 'Yes', 'no': 'No'})

    if 'ZIP code' in out.columns:
        out['ZIP code'] = out['ZIP code'].astype(str).str.extract(r'(\d{5})', expand=False)

    for date_col in ['Date received', 'Date sent to company']:
        if date_col in out.columns:
            out[date_col] = pd.to_datetime(out[date_col], errors='coerce')

    out = out.drop_duplicates()
    if ID_COLUMN in out.columns:
        out = out.drop_duplicates(subset=[ID_COLUMN], keep='first')
        out = out.sort_values(ID_COLUMN).reset_index(drop=True)

    if is_training:
        target_clean = out[TARGET_COLUMN].astype(str).str.strip().str.lower()
        out['target'] = target_clean.map({'yes': 1, 'no': 0})
        out = out[out['target'].isin([0, 1])].copy()
        out['target'] = out['target'].astype(np.int8)

    return out.reset_index(drop=True)


def add_minimal_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create a small set of explicit structured features for the assignment."""
    out = df.copy()

    narrative = out['Consumer complaint narrative'].fillna('')
    out['narrative_present'] = narrative.ne('').astype(np.int8)
    out['narrative_len'] = narrative.str.len().fillna(0).astype(np.float32)

    received_dt = pd.to_datetime(out['Date received'], errors='coerce')
    sent_dt = pd.to_datetime(out['Date sent to company'], errors='coerce')
    out['days_to_company'] = (sent_dt - received_dt).dt.days.astype('float32')
    out['month_received'] = received_dt.dt.month.astype('float32')
    out['weekday_received'] = received_dt.dt.dayofweek.astype('float32')

    for col in CATEGORICAL_COLUMNS:
        out[col] = out[col].fillna('Missing').astype(str)

    for col in NUMERIC_COLUMNS:
        out[col] = out[col].fillna(-1).astype('float32')

    return out

## Data Loading And Initial Checks

The raw files are loaded first and then passed through the cleaning and feature-generation steps. The displayed samples provide a quick check that the target, categorical fields, and derived numeric variables look sensible before model fitting begins.

In [4]:
train_raw = pd.read_csv(TRAIN_PATH)
inference_raw = pd.read_csv(INFERENCE_PATH)

print('Raw training shape:', train_raw.shape)
print('Raw inference shape:', inference_raw.shape)
display(train_raw.head(2))

Raw training shape: (321430, 18)
Raw inference shape: (100, 18)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2015-12-17,Mortgage,Other mortgage,"Loan modification,collection,foreclosure",NaN,NaN,NaN,Ocwen Financial Corporation,ME,04419,"Older American, Servicemember",NaN,Referral,2015-12-23,Closed with explanation,Yes,Yes,1705202
1,2015-05-27,Mortgage,Other mortgage,"Loan modification,collection,foreclosure",NaN,NaN,Company chooses not to provide a public response,WELLS FARGO & COMPANY,FL,33615,Servicemember,NaN,Referral,2015-05-29,Closed with explanation,Yes,No,1394282


In [5]:
train_clean = add_minimal_features(clean_complaints_df(train_raw, is_training=True))
inference_clean = add_minimal_features(clean_complaints_df(inference_raw, is_training=False))

print('Clean training rows:', len(train_clean))
print('Clean inference rows:', len(inference_clean))
print('Target distribution:')
display(train_clean['target'].value_counts().rename_axis('target').to_frame('rows'))
display(train_clean[MODEL_COLUMNS + ['target']].head(3))

C:\Users\conno\AppData\Local\Temp\ipykernel_38492\2551056869.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = out.select_dtypes(include=['object']).columns.tolist()


Clean training rows: 321430
Clean inference rows: 100
Target distribution:


C:\Users\conno\AppData\Local\Temp\ipykernel_38492\2551056869.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = out.select_dtypes(include=['object']).columns.tolist()


,rows
target,
0,257436
1,63994


,Product,Sub-product,Issue,Sub-issue,Company,State,Tags,Consumer consent provided?,Submitted via,Company response to consumer,Company public response,Timely response?,days_to_company,month_received,weekday_received,narrative_present,narrative_len,target
0,Bank account or service,(CD) Certificate of deposit,Deposits and withdrawals,Missing,AMERICAN EXPRESS COMPANY,CA,Missing,Missing,Web,Closed with explanation,Missing,Yes,0.0,1.0,2.0,0.0,0.0,0
1,Bank account or service,Checking account,Deposits and withdrawals,Missing,WELLS FARGO & COMPANY,FL,Missing,Missing,Web,Closed with explanation,Missing,Yes,0.0,1.0,2.0,0.0,0.0,1
2,Debt collection,I do not know,Disclosure verification of debt,Right to dispute notice not received,ENCORE CAPITAL GROUP INC.,AR,Missing,Missing,Web,Closed with explanation,Missing,Yes,1.0,1.0,2.0,0.0,0.0,0


## Deterministic Dataset Subsets

The assignment requires a size sweep, so the notebook samples rows deterministically rather than taking arbitrary slices. Using the same seed at every fraction ensures that later CPU and GPU runs are built from comparable subsets.

In [6]:
def subset_row_positions(n_rows: int, fraction: float, seed: int = RANDOM_STATE) -> list[int]:
    """Pick deterministic row positions for the size sweep."""
    if not 0 < fraction <= 1:
        raise ValueError('fraction must be in the interval (0, 1].')

    target_rows = max(1, int(round(n_rows * fraction)))
    if target_rows >= n_rows:
        return list(range(n_rows))

    rng = Random(seed)
    return sorted(rng.sample(range(n_rows), target_rows))



def subset_rows(df, fraction: float, seed: int = RANDOM_STATE):
    positions = subset_row_positions(len(df), fraction, seed)
    return df.iloc[positions].reset_index(drop=True)


ACTIVE_FRACTION = 1.00
train_subset = subset_rows(train_clean, ACTIVE_FRACTION)
inference_subset = subset_rows(inference_clean, ACTIVE_FRACTION)

print('Active subset fraction:', ACTIVE_FRACTION)
print('Training rows after subsetting:', len(train_subset))
print('Inference rows after subsetting:', len(inference_subset))


Active subset fraction: 1.0
Training rows after subsetting: 321430
Inference rows after subsetting: 100


## Train-Validation Split And Encoded Features

The cleaned subset is divided into training and validation partitions with a custom stratified split. After that, the selected predictors are one-hot encoded and the columns are aligned across training, validation, and external inference data so the model always sees a consistent feature matrix.

In [7]:
def encode_features_cpu(df: pd.DataFrame) -> pd.DataFrame:
    encoded = pd.get_dummies(df[MODEL_COLUMNS].copy(), columns=CATEGORICAL_COLUMNS, dtype=np.int8)
    encoded = encoded.reindex(sorted(encoded.columns), axis=1)
    return encoded



def align_to_reference_columns(encoded_df: pd.DataFrame, reference_columns) -> pd.DataFrame:
    aligned = encoded_df.copy()
    for col in reference_columns:
        if col not in aligned.columns:
            aligned[col] = 0
    extra_cols = [col for col in aligned.columns if col not in reference_columns]
    if extra_cols:
        aligned = aligned.drop(columns=extra_cols)
    return aligned.loc[:, reference_columns]



def stratified_train_validation_indices(labels, validation_size: float, seed: int = RANDOM_STATE):
    label_positions = {}
    for idx, value in enumerate(labels):
        label_positions.setdefault(int(value), []).append(idx)

    rng = Random(seed)
    train_idx = []
    val_idx = []

    for label in sorted(label_positions):
        positions = label_positions[label][:]
        rng.shuffle(positions)

        if len(positions) <= 1:
            val_count = 0
        else:
            val_count = int(round(len(positions) * validation_size))
            val_count = max(1, min(len(positions) - 1, val_count))

        val_idx.extend(sorted(positions[:val_count]))
        train_idx.extend(sorted(positions[val_count:]))

    return sorted(train_idx), sorted(val_idx)



def prepare_cpu_datasets(train_df: pd.DataFrame, inference_df: pd.DataFrame):
    target = [int(value) for value in train_df['target'].tolist()]
    train_idx, val_idx = stratified_train_validation_indices(target, VALIDATION_SIZE, seed=RANDOM_STATE)

    train_part = train_df.iloc[train_idx].reset_index(drop=True)
    val_part = train_df.iloc[val_idx].reset_index(drop=True)

    X_train = encode_features_cpu(train_part)
    X_val = encode_features_cpu(val_part)
    X_inference = encode_features_cpu(inference_df)

    reference_columns = list(X_train.columns)
    X_val = align_to_reference_columns(X_val, reference_columns)
    X_inference = align_to_reference_columns(X_inference, reference_columns)

    y_train = train_part['target'].to_numpy(dtype=np.int8, copy=False)
    y_val = val_part['target'].to_numpy(dtype=np.int8, copy=False)

    return {
        'train_part': train_part,
        'val_part': val_part,
        'X_train': X_train,
        'X_val': X_val,
        'X_inference': X_inference,
        'y_train': y_train,
        'y_val': y_val,
        'feature_columns': reference_columns,
    }



datasets = prepare_cpu_datasets(train_subset, inference_subset)
print('Training matrix shape:', datasets['X_train'].shape)
print('Validation matrix shape:', datasets['X_val'].shape)
print('Inference matrix shape:', datasets['X_inference'].shape)
print('Encoded feature count:', len(datasets['feature_columns']))


C:\Users\conno\AppData\Local\Temp\ipykernel_38492\1046238871.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aligned[col] = 0


Training matrix shape: (257144, 3375)
Validation matrix shape: (64286, 3375)
Inference matrix shape: (100, 3375)
Encoded feature count: 3375


## Model Fitting, Validation, And Inference

This section trains the CPU XGBoost model, evaluates it on the held-out validation set, and generates predictions for the separate inference file. The outputs establish both the predictive behaviour of the baseline and the exact artifacts that will later be compared with the GPU implementation.

In [8]:
def build_cpu_model(y_train: np.ndarray) -> XGBClassifier:
    negatives = int((y_train == 0).sum())
    positives = int((y_train == 1).sum())
    scale_pos_weight = negatives / max(positives, 1)

    return XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.80,
        colsample_bytree=0.80,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        tree_method='hist',
        device='cpu',
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    )



def evaluate_binary(y_true: np.ndarray, positive_scores: np.ndarray, threshold: float = PREDICTION_THRESHOLD) -> dict:
    predictions = (positive_scores >= threshold).astype(np.int8)

    tp = int(np.logical_and(predictions == 1, y_true == 1).sum())
    tn = int(np.logical_and(predictions == 0, y_true == 0).sum())
    fp = int(np.logical_and(predictions == 1, y_true == 0).sum())
    fn = int(np.logical_and(predictions == 0, y_true == 1).sum())

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return {
        'threshold': threshold,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': predictions,
    }


cpu_model = build_cpu_model(datasets['y_train'])
cpu_model.fit(datasets['X_train'], datasets['y_train'])

val_scores = cpu_model.predict_proba(datasets['X_val'])[:, 1]
val_metrics = evaluate_binary(datasets['y_val'], val_scores, threshold=PREDICTION_THRESHOLD)
inference_scores = cpu_model.predict_proba(datasets['X_inference'])[:, 1]
inference_predictions = (inference_scores >= PREDICTION_THRESHOLD).astype(np.int8)

metrics_table = pd.DataFrame([
    {
        'accuracy': val_metrics['accuracy'],
        'precision': val_metrics['precision'],
        'recall': val_metrics['recall'],
        'f1': val_metrics['f1'],
        'validation_rows': len(datasets['y_val']),
        'inference_rows': len(inference_predictions),
    }
]).round(4)

display(metrics_table)
display(pd.DataFrame({
    'Complaint ID': inference_subset[ID_COLUMN],
    'cpu_probability': inference_scores,
    'cpu_prediction': inference_predictions,
}).head(10))


,accuracy,precision,recall,f1,validation_rows,inference_rows
0,0.564,0.2677,0.6859,0.3851,64286,100


,Complaint ID,cpu_probability,cpu_prediction
0,2271627,0.598486,1
1,2274992,0.614718,1
2,2289332,0.554243,1
3,2289336,0.333627,0
4,2312355,0.470323,0
5,2331272,0.481389,0
6,2340546,0.541021,1
7,2356986,0.675362,1
8,2372244,0.727558,1
9,2379919,0.536719,1


## CPU Output Bundle

The notebook stores a compact summary of the CPU run: row counts, encoded columns, prediction lengths, validation metrics, and probability vectors.

That bundle makes it straightforward to verify that the GPU notebook is solving the same task and producing closely matched results.


In [9]:
def build_cpu_artifact_bundle() -> dict:
    return {
        'fraction': ACTIVE_FRACTION,
        'train_rows_after_cleaning': int(len(train_subset)),
        'inference_rows_after_cleaning': int(len(inference_subset)),
        'feature_columns': list(datasets['feature_columns']),
        'validation_prediction_length': int(len(val_metrics['predictions'])),
        'inference_prediction_length': int(len(inference_predictions)),
        'metrics': {
            'accuracy': float(val_metrics['accuracy']),
            'precision': float(val_metrics['precision']),
            'recall': float(val_metrics['recall']),
            'f1': float(val_metrics['f1']),
        },
        'validation_probabilities': val_scores.tolist(),
        'inference_probabilities': inference_scores.tolist(),
    }


cpu_artifact_bundle = build_cpu_artifact_bundle()
cpu_artifact_overview = pd.DataFrame([
    {
        'fraction': cpu_artifact_bundle['fraction'],
        'train_rows_after_cleaning': cpu_artifact_bundle['train_rows_after_cleaning'],
        'inference_rows_after_cleaning': cpu_artifact_bundle['inference_rows_after_cleaning'],
        'encoded_feature_count': len(cpu_artifact_bundle['feature_columns']),
        'validation_prediction_length': cpu_artifact_bundle['validation_prediction_length'],
        'inference_prediction_length': cpu_artifact_bundle['inference_prediction_length'],
        'accuracy': cpu_artifact_bundle['metrics']['accuracy'],
        'precision': cpu_artifact_bundle['metrics']['precision'],
        'recall': cpu_artifact_bundle['metrics']['recall'],
        'f1': cpu_artifact_bundle['metrics']['f1'],
    }
]).round(4)

display(cpu_artifact_overview)


,fraction,train_rows_after_cleaning,inference_rows_after_cleaning,encoded_feature_count,validation_prediction_length,inference_prediction_length,accuracy,precision,recall,f1
0,1.0,321430,100,3375,64286,100,0.564,0.2677,0.6859,0.3851


## Benchmark Sweep

The final stage repeats the pipeline at 10%, 25%, 50%, 75%, and 100% of the cleaned data.

Training time, validation inference time, and external inference time are recorded separately, with three repeats per size, so the later analysis can explain where GPU acceleration helps and where it does not.


In [10]:
def time_block(func):
    start = perf_counter()
    result = func()
    elapsed = perf_counter() - start
    return result, elapsed



def summarize_timings(values) -> tuple[float, float]:
    if len(values) == 1:
        return float(values[0]), 0.0
    return float(mean(values)), float(stdev(values))



def run_cpu_pipeline_for_fraction(fraction: float, repeats: int = BENCHMARK_REPEATS) -> dict:
    sampled_train = subset_rows(train_clean, fraction)
    sampled_inference = subset_rows(inference_clean, fraction)

    train_timings = []
    validation_timings = []
    inference_timings = []
    last_metrics = None
    last_datasets = None

    for _ in range(repeats):
        sampled_datasets = prepare_cpu_datasets(sampled_train, sampled_inference)
        model = build_cpu_model(sampled_datasets['y_train'])

        _, train_seconds = time_block(lambda: model.fit(sampled_datasets['X_train'], sampled_datasets['y_train']))
        validation_scores_local, validation_seconds = time_block(lambda: model.predict_proba(sampled_datasets['X_val'])[:, 1])
        _, inference_seconds = time_block(lambda: model.predict_proba(sampled_datasets['X_inference'])[:, 1])

        train_timings.append(train_seconds)
        validation_timings.append(validation_seconds)
        inference_timings.append(inference_seconds)
        last_metrics = evaluate_binary(sampled_datasets['y_val'], validation_scores_local)
        last_datasets = sampled_datasets

    train_mean, train_std = summarize_timings(train_timings)
    validation_mean, validation_std = summarize_timings(validation_timings)
    inference_mean, inference_std = summarize_timings(inference_timings)

    return {
        'fraction': fraction,
        'train_rows': len(sampled_train),
        'validation_rows': len(last_datasets['y_val']),
        'inference_rows': len(sampled_inference),
        'benchmark_repeats': repeats,
        'train_seconds_mean': train_mean,
        'train_seconds_std': train_std,
        'validation_seconds_mean': validation_mean,
        'validation_seconds_std': validation_std,
        'inference_seconds_mean': inference_mean,
        'inference_seconds_std': inference_std,
        'accuracy': last_metrics['accuracy'],
        'precision': last_metrics['precision'],
        'recall': last_metrics['recall'],
        'f1': last_metrics['f1'],
    }



def run_cpu_benchmark() -> pd.DataFrame:
    results = [run_cpu_pipeline_for_fraction(fraction) for fraction in SUBSET_FRACTIONS]
    return pd.DataFrame(results)[BENCHMARK_COLUMNS]


cpu_benchmark_results = run_cpu_benchmark()
display(cpu_benchmark_results.round(4))


C:\Users\conno\AppData\Local\Temp\ipykernel_38492\1046238871.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aligned[col] = 0
C:\Users\conno\AppData\Local\Temp\ipykernel_38492\1046238871.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aligned[col] = 0
C:\Users\conno\AppData\Local\Temp\ipykernel_38492\1046238871.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis

,fraction,train_rows,validation_rows,inference_rows,benchmark_repeats,train_seconds_mean,train_seconds_std,validation_seconds_mean,validation_seconds_std,inference_seconds_mean,inference_seconds_std,accuracy,precision,recall,f1
0,0.10,32143,6429,10,3,7.5436,0.0222,0.1750,0.0011,0.1636,0.0512,0.5923,0.2576,0.5575,0.3524
1,0.25,80358,16072,25,3,16.1709,0.3687,0.3748,0.0504,0.1626,0.0216,0.5735,0.2648,0.6359,0.3739
2,0.50,160715,32143,50,3,30.7553,1.1462,0.5420,0.0320,0.2095,0.0443,0.5658,0.2668,0.6694,0.3815
3,0.75,241072,48214,75,3,46.2221,1.4436,0.8463,0.0418,0.2111,0.0486,0.5629,0.2683,0.6870,0.3859
4,1.00,321430,64286,100,3,69.2820,2.8952,1.2309,0.0539,0.2621,0.0837,0.5640,0.2677,0.6859,0.3851
